<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Implementación con Transforms</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Después de entender la lógica del augmentation, toca ver cómo integrarlo en el flujo real de datos. Este notebook se centra en implementarlo con transforms dentro del pipeline de PyTorch.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Una idea útil solo se vuelve reproducible cuando puede incorporarse de forma clara al procesamiento de datos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A situar el augmentation dentro de un pipeline de transforms y a entender su papel operativo durante el entrenamiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Una buena implementación debe ser legible, reutilizable y coherente con el resto del pipeline de datos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Implementar augmentation con transforms significa encapsular cambios de datos en una secuencia aplicable a cada muestra.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Creación de una imagen de ejemplo
"""Este bloque crea una imagen de ejemplo para aplicar transformaciones con torchvision.transforms.
Este bloque importa "Image" de "PIL" para crear imágenes y "torchvision.transforms" para aplicar data augmentation.

Aplicamos la función "Image.new()" a la variable "imagen" para crear una imagen uniforme con un color específico.
El argumento "RGB" indica que la imagen tendrá tres canales de color: rojo, verde y azul.
El argumento "(128, 128)" especifica el tamaño de la imagen en píxeles; en este caso, 128 píxeles de ancho y 128 píxeles de alto.
El parámetro "color=" especifica el color de fondo de la imagen; en este caso, un color RGB con valores (180, 140, 220).

La variable "imagen" contiene la imagen original.
El uso de una imagen uniforme permite observar algunos cambios, aunque no muestra el efecto de todas las transformaciones sobre detalles visuales."""

from PIL import Image
import torchvision.transforms as T

imagen = Image.new("RGB", (128, 128), color=(180, 140, 220))

imagen # Muestra la imagen original.


In [ ]:
# Bloque 2 — Definición de transformaciones con transforms
"""Este bloque define una secuencia de transformaciones utilizando torchvision.transforms.
Aplicamos la clase "T.Compose()" a la variable "pipeline" para definir una secuencia de transformaciones que se aplicará a las imágenes.

La clase "T.RandomRotation()" permite rotar la imagen aleatoriamente dentro de un rango de grados especificado.
El parámetro "degrees=" especifica el rango de grados dentro del cual se puede rotar la imagen aleatoriamente.

La clase "T.RandomResizedCrop()" permite recortar aleatoriamente una región de la imagen y redimensionarla a un tamaño específico.
El parámetro "size=" especifica el tamaño de la imagen resultante.
El parámetro "scale=" especifica el rango de proporciones del área original para el recorte aleatorio.

La clase "T.ColorJitter()" permite ajustar el brillo, contraste, saturación y matiz de la imagen.
El parámetro "brightness=" especifica el rango de ajuste del brillo de la imagen.
El parámetro "contrast=" especifica el rango de ajuste del contraste de la imagen.

La clase "T.ToTensor()" convierte la imagen en un tensor de PyTorch.
  
La clase "T.Normalize()" permite normalizar la imagen utilizando una media y desviación estándar especificadas.
El parámetro "mean=" especifica la media utilizada para normalizar cada canal de la imagen.
El parámetro "std=" especifica la desviación estándar utilizada para normalizar cada canal de la imagen.
El canal de la imagen se refiere a los componentes de color individuales (rojo, verde y azul) que se normalizan utilizando la media y 
desviación estándar especificadas.

Las transformaciones se aplican en secuencia; algunas generan variaciones y otras preparan la imagen para el modelo.
La variable "pipeline" almacena la secuencia completa de transformaciones."""

pipeline = T.Compose([
    T.RandomRotation(degrees=20),
    T.RandomResizedCrop(size=128, scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.2, 0.2, 0.2])
])

pipeline # Muestra la secuencia de transformaciones definida en la variable "pipeline".


In [ ]:
# Bloque 3 — Aplicación de la secuencia de transforms
"""Este bloque aplica la secuencia de transformaciones definida en pipeline.
Asignamos la llamada a la instancia de la clase a la variable "imagen_transformada" mediante la expresión "pipeline(imagen)".
El argumento "imagen" es la imagen original sobre la cual se aplicarán las transformaciones.
La variable "imagen_transformada" contiene la imagen transformada, convertida en tensor y normalizada.
Cada ejecución puede producir una variación distinta debido a la aleatoriedad de algunas transformaciones."""

imagen_transformada = pipeline(imagen)

imagen_transformada # Muestra la imagen transformada, convertida en tensor y normalizada.


In [ ]:
# Bloque 4 — Transformación determinista adicional
"""Este bloque aplica una transformación determinista utilizando transforms funcionales.
Este bloque importa "torchvision.transforms.functional" para poder aplicar transformaciones funcionales deterministas a las imágenes.

Aplicamos la función "F.adjust_hue()" a la variable "imagen_hue" para modificar el matiz de la imagen.
La variable "imagen_desnormalizada" recupera el intervalo de píxeles [0, 1] antes de ajustar el matiz.
El parámetro "hue_factor=" controla la cantidad de ajuste de matiz que se aplicará a la imagen.
Un valor positivo aumenta el matiz, mientras que un valor negativo lo disminuye.

La operación modifica el matiz de la imagen transformada sin alterar su estructura.
La desnormalización invierte la media 0.5 y la desviación 0.2 usadas en el pipeline; clamp limita posibles errores de redondeo.
La variable "imagen_hue" contiene el tensor resultante en el intervalo [0, 1]."""

import torchvision.transforms.functional as F

imagen_desnormalizada = (imagen_transformada * 0.2 + 0.5).clamp(0, 1)
imagen_hue = F.adjust_hue(imagen_desnormalizada, hue_factor=0.1)

imagen_hue # Muestra el tensor resultante del ajuste de matiz.


In [ ]:
# Bloque 5 — Resultados finales
"""Este bloque organiza los resultados del proceso de implementación con transforms.
La variable "resultado" almacena en forma de diccionario el tensor transformado y el tensor con ajuste de matiz, convertidos en listas.
La estructura permite visualizar el resultado normalizado de la secuencia y el ajuste adicional en el intervalo [0, 1]."""

resultado = {
    "tensor_transformado": imagen_transformada.tolist(),
    "tensor_hue": imagen_hue.tolist()
}

resultado # Muestra el diccionario con los resultados finales de las transformaciones.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo convertir el augmentation en una parte operativa del pipeline de datos.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Integrar bien las transforms facilita repetir experimentos y mantener consistencia en el entrenamiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Aplicar augmentations de manera dispersa o manual hasta perder trazabilidad sobre qué se está haciendo a cada muestra.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en las Redes Convolucionales para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en las Redes Convolucionales y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">09_Redes_convolucionales/01_Las_redes_convolucionales.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">